In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
from peft import PeftConfig, PeftModel
from huggingface_hub import login
import torch

login(token="HF_TOKEN")
!huggingface-cli whoami

device = "cuda"

# Prompts

In [ ]:
prompt_test = "Write a sentence in Bavarian dialect."
prompt_test_de = "Schreibe einen Satz auf Bairisch."
prompt_test_ba = "Schreib an Satz auf Boarisch."


prompt_qa_zero = """Generate a question with an indirect answer in Bavarian dialect. 
Label the polarity of the answer with this label set:
1 = a clear yes
2 = a clear no
3 = yes, subject to some conditions
4 = neither yes or no (middle)
5 = other, the answer does not match the question
6 = not sure whether yes or no because context is missing
"""

prompt_qa_zero_de = """Generiere eine Frage mit einer indirekten Antwort auf Bairisch.
Annotiere die Polarität der Antwort mit diesem Labelset:
1 = ein klares Ja
2 = ein klares Nein
3 = ja, unter bestimmten Bedingungen
4 = weder ja noch nein (Mitte)
5 = anders, die Antwort passt nicht zur Frage
6 = nicht sicher, ob ja oder nein, weil der Kontext fehlt
"""

prompt_qa_few = """Generate question, an indirect answer, and a correct label following the pattern in Bavarian dialect.
Do not use the words Yes or No or other direct answers. 

Use the following label set:
1 = a clear yes
2 = a clear no
3 = yes, subject to some conditions
4 = neither yes or no (middle)
5 = other, the answer does not match the question
6 = not sure whether yes or no because context is missing

Examples:
Host du scho vom Finanzamt gheard?\tI hob an Briaf vo eana griagt.\t1
Host du mei Grawattn gfundn?\tI hob ned danach gsuacht.\t2
Hoit da Herr Schmid de Vorlesung nächsts Joar?\tDer is nächsts Joar nimma an da Uni.\t2
"""

prompt_qa_few_de = """Erstelle eine Frage, eine indirekte Antwort und ein korrektes Label nach dem Muster auf Bairisch.
Verwende nicht die Worte Ja oder Nein oder andere direkte Antworten. 

Verwende diese Labels:
1 = ein klares Ja
2 = ein klares Nein
3 = ja, unter bestimmten Bedingungen
4 = weder ja noch nein (Mitte)
5 = anders, die Antwort passt nicht zur Frage
6 = nicht sicher, ob ja oder nein, weil der Kontext fehlt

Beispiele:
Host du scho vom Finanzamt gheard?\tI hob an Briaf vo eana griagt.\t1
Host du mei Grawattn gfundn?\tI hob ned danach gsuacht.\t2
Hoit da Herr Schmid de Vorlesung nächsts Joar?\tDer is nächsts Joar nimma an da Uni.\t2
"""

# LLMs

### EuroLLM

#### EuroLLM-1.7B-Instruct
Code von https://huggingface.co/utter-project/EuroLLM-1.7B-Instruct

In [ ]:
model_id_euro_small = "utter-project/EuroLLM-1.7B-Instruct"

# load from huggingface
model_euro_small = AutoModelForCausalLM.from_pretrained(model_id_euro_small)
tokenizer_euro_small = AutoTokenizer.from_pretrained(model_id_euro_small)

In [ ]:
def reformat_prompt_euro_small(prompt):
    return f"<|im_start|>system\n<|im_end|>\n<|im_start|>user\n{prompt} <|im_end|>\n<|im_start|>assistant\n"

In [ ]:
# check general Bavarian capabilities
inputs = tokenizer_euro_small(reformat_prompt_euro_small(prompt_test), return_tensors="pt")
outputs = model_euro_small.generate(**inputs, max_new_tokens=128)

generated_text = tokenizer_euro_small.decode(outputs[0], skip_special_tokens=True)
print(generated_text.split("\n")[-1]) # only return the generated response

Setting `pad_token_id` to `eos_token_id`:4 for open-end generation.


Das ist Deutsch, so hab ich Deutsch gelernt, und das ist gut Deutsch.


In [ ]:
# zero-shot prompting
inputs = tokenizer_euro_small(reformat_prompt_euro_small(prompt_qa_zero), return_tensors="pt")
outputs = model_euro_small.generate(**inputs, max_new_tokens=128)

generated_text = tokenizer_euro_small.decode(outputs[0], skip_special_tokens=True)
print(generated_text.split("\n")[-1]) 
# print(generated_text)

Setting `pad_token_id` to `eos_token_id`:4 for open-end generation.


What are the advantages of using a data warehouse in a business?


In [ ]:
# few-shot prompting
inputs = tokenizer_euro_small(reformat_prompt_euro_small(prompt_qa_few), return_tensors="pt")
outputs = model_euro_small.generate(**inputs, max_new_tokens=128)

generated_text = tokenizer_euro_small.decode(outputs[0], skip_special_tokens=True)
print(generated_text.split("\n")[-1])

Setting `pad_token_id` to `eos_token_id`:4 for open-end generation.


a clear yes


#### EuroLLM-9B-Instruct
Code from https://huggingface.co/utter-project/EuroLLM-9B-Instruct

In [ ]:
model_id_euro_large = "utter-project/EuroLLM-9B-Instruct"

# load from huggingface
model_euro_large = AutoModelForCausalLM.from_pretrained(model_id_euro_large)
tokenizer_euro_large = AutoTokenizer.from_pretrained(model_id_euro_large)

Loading checkpoint shards: 100%|██████████| 4/4 [07:01<00:00, 105.45s/it]


In [3]:
# default EuroLLM system prompt
system_euro_large = "You are EuroLLM --- an AI assistant specialized in European languages that provides safe, educational and helpful answers."

def reformat_prompt_euro_large(prompt, system=system_euro_large):
    message = [
        {"role": "system", "content": system},
        {"role": "user", "content": prompt},
    ]
    return message

In [ ]:
# check general Bavarian capabilities
inputs = tokenizer_euro_large.apply_chat_template(reformat_prompt_euro_large(prompt_test), tokenize=True, add_generation_prompt=True, return_tensors="pt")
outputs = model_euro_large.generate(inputs, max_new_tokens=128) 

generated_text = tokenizer_euro_large.decode(outputs[0], skip_special_tokens=True)
# print(generated_text.split("\n")[-1]) # only return the generated response
print(generated_text)

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:4 for open-end generation.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


system
You are EuroLLM --- an AI assistant specialized in European languages that provides safe, educational and helpful answers.
user
Write a sentence in Bavarian.
assistant
I bin do in Land, wo's Bier so g'scheit is. (I am in the land where beer is so good.)


In [ ]:
# zero-shot prompting
inputs = tokenizer_euro_large.apply_chat_template(reformat_prompt_euro_large(prompt_qa_zero), tokenize=True, add_generation_prompt=True, return_tensors="pt")
outputs = model_euro_large.generate(inputs, max_new_tokens=128, max_time=1800) # limit time to 30 minutes because it takes too long to generate

generated_text = tokenizer_euro_large.decode(outputs[0], skip_special_tokens=True)
# print(generated_text.split("\n")[-1]) # only return the generated response
print(generated_text)

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:4 for open-end generation.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


system
You are EuroLLM --- an AI assistant specialized in European languages that provides safe, educational and helpful answers.
user
Generate a question with an indirect answer in Bavarian. 
Label the polarity of the answer with this label set:
1 = a clear yes
2 = a clear no
3 = yes, subject to some conditions
4 = neither yes or no (middle)
5 = other, the answer does not match the question
6 = not sure whether yes or no because context is missing
assistant
Kannst du mir helfen, einen Weg durch den Wald zu


In [ ]:
# few-shot prompting
inputs = tokenizer_euro_large.apply_chat_template(reformat_prompt_euro_large(prompt_qa_few), tokenize=True, add_generation_prompt=True, return_tensors="pt")
outputs = model_euro_large.generate(inputs, max_new_tokens=128, max_time=1800)

generated_text = tokenizer_euro_large.decode(outputs[0], skip_special_tokens=True)
# print(generated_text.split("\n")[-1]) # only return the generated response
print(generated_text)

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:4 for open-end generation.


system
You are EuroLLM --- an AI assistant specialized in European languages that provides safe, educational and helpful answers.
user
Generate question, an indirect answer, and a correct label following the pattern in Bavarian dialect.
Do not use the words Yes or No or other direct answers. 

Use the following label set:
1 = a clear yes
2 = a clear no
3 = yes, subject to some conditions
4 = neither yes or no (middle)
5 = other, the answer does not match the question
6 = not sure whether yes or no because context is missing

Examples:
Host du scho vom Finanzamt gheard?	I hob an Briaf vo eana griagt.	1
Host du mei Grawattn gfundn?	I hob ned danach gsuacht.	2
Hoit da Herr Schmid de Vorlesung nächsts Joar?	Der is nächsts Joar nimma an da Uni.	2
assistant
Hast denn da Frida schon am Bus-Halt gwunn


### LLäMmlein
Code from https://huggingface.co/LSX-UniWue/LLaMmlein_1B

Note: not instruction-tuned, so it cannot follow prompts

In [ ]:
model_id_llammlein = "LSX-UniWue/LLaMmlein_1B"

# load from huggingface
model_llammlein = AutoModelForCausalLM.from_pretrained(model_id_llammlein)
tokenizer_llammlein = AutoTokenizer.from_pretrained(model_id_llammlein)

In [ ]:
# check general Bavarian capabilities
inputs = tokenizer_llammlein(prompt_test_de, return_tensors="pt")
outputs = model_llammlein.generate(**inputs, max_length=128)

generated_text = tokenizer_llammlein.decode(outputs[0], skip_special_tokens=True)
print(generated_text)

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


Schreibe einen Satz auf Bairisch.
Die meisten Menschen in Bayern sprechen Bairisch.
Bairisch ist eine Sprache, die in Bayern gesprochen wird.
Bairisch ist eine Sprache, die in Bayern gesprochen wird. Sie ist eine Variante des Deutschen.
Bairisch ist eine Sprache, die in Bayern gesprochen wird. Sie ist eine Variante des Deutschen. Sie ist eine Variante des Deutschen.
Bairisch ist eine Sprache, die in Bayern gesprochen wird. Sie ist eine Variante des Deutschen. Sie ist eine Variante des Deutschen.
Bairisch ist eine Sprache, die in Bayern gesprochen wird. Sie ist


In [ ]:
# zero-shot prompting
inputs = tokenizer_llammlein(prompt_qa_zero_de, return_tensors="pt")
outputs = model_llammlein.generate(**inputs, max_length=128)

generated_text = tokenizer_llammlein.decode(outputs[0], skip_special_tokens=True)
print(generated_text)

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


Generiere eine Frage mit einer indirekten Antwort auf Bairisch.
Annotiere die Polarität der Antwort mit diesem Labelset:
1 = ein klares Ja
2 = ein klares Nein
3 = ja, unter bestimmten Bedingungen
4 = weder ja noch nein (Mitte)
5 = anders, die Antwort passt nicht zur Frage
6 = nicht sicher, ob ja oder nein, weil der Kontext fehlt
7 = nicht sicher, ob ja oder nein, weil der Kontext fehlt (Mitte)
8 = nicht sicher, ob ja oder nein, weil der Kontext fehlt (Ende)
9 = nicht sicher,


In [ ]:
# few-shot prompting
inputs = tokenizer_llammlein(prompt_qa_few_de, return_tensors="pt")
outputs = model_llammlein.generate(**inputs, max_length=256)

generated_text = tokenizer_llammlein.decode(outputs[0], skip_special_tokens=True)
print(generated_text)

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


Erstelle eine Frage, eine indirekte Antwort und ein korrektes Label nach dem Muster auf Bayerisch.
Verwende nicht die Worte Ja oder Nein oder andere direkte Antworten. 

Verwende diese Labels:
1 = ein klares Ja
2 = ein klares Nein
3 = ja, unter bestimmten Bedingungen
4 = weder ja noch nein (Mitte)
5 = anders, die Antwort passt nicht zur Frage
6 = nicht sicher, ob ja oder nein, weil der Kontext fehlt

Beispiele:
Host du scho vom Finanzamt gheard?	I hob an Briaf vo eana griagt.	1
Host du mei Grawattn gfundn?	I hob ned danach gsuacht.	2
Hoit da Herr Schmid de Vorlesung nächsts Joar?	Der is nächsts Joar nimma an da Uni.	2
Hast du schon mal was von der EU gehört?	I hob an Briaf vo eana griagt.	3
Hast du schon mal was von der EU gehört?	I hob an Briaf vo eana griagt.	3
Hast du schon mal was von der EU


#### Bavarian adapter: Betzerl
Code from https://huggingface.co/LSX-UniWue/Betzerl_1B_wiki_preview

In [ ]:
model_id_llammlein = "LSX-UniWue/LLaMmlein_1B"
adapter_name_betzerl = "LSX-UniWue/Betzerl_1B_wiki_preview"

# load adapter for LLäMmlein
config = PeftConfig.from_pretrained(adapter_name_betzerl)

base_model = model = AutoModelForCausalLM.from_pretrained(
    model_id_llammlein,
    torch_dtype=torch.bfloat16,
    device_map=device,
)
base_model.resize_token_embeddings(32001)

model_betzerl = PeftModel.from_pretrained(base_model, adapter_name_betzerl)
model_betzerl = model_betzerl.to(device)
tokenizer_betzerl = AutoTokenizer.from_pretrained(adapter_name_betzerl)

c:\Users\Winkl\Music\msc-repo\masterenv\lib\site-packages\huggingface_hub\file_download.py:142: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in E:\LLMs\hub\models--LSX-UniWue--Betzerl_1B_wiki_preview. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covaria

In [ ]:
# check general Bavarian capabilities

print("\nStandard German prompt:\n")
inputs = tokenizer_betzerl(prompt_test_ba, return_tensors="pt").to(device)
outputs = model_betzerl.generate(**inputs, max_length=128)

generated_text = tokenizer_betzerl.decode(outputs[0], skip_special_tokens=True)
print(generated_text, "\n")

print("\nBavarian prompt:\n")
inputs = tokenizer_betzerl(prompt_test_ba, return_tensors="pt").to(device)
outputs = model_betzerl.generate(**inputs, max_length=128)

generated_text = tokenizer_betzerl.decode(outputs[0], skip_special_tokens=True)
print(generated_text)

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.



Standard German prompt:



Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


Schreib an Satz auf Boarisch.

Schreib an Satz auf Boarisch.

Schreib an Satz auf Boarisch.

Schreib an Satz auf Boarisch.

Schreib an Satz auf Boarisch.

Schreib an Satz auf Boarisch.

Schreib an Satz auf Boarisch.

Schreib an Satz auf Boarisch.

Schreib an Satz auf Boarisch.

Schreib an Satz auf Boarisch.

Schreib an Satz auf Boarisch.

Schreib an Satz auf Boarisch.

Schreib an Satz auf Boarisch 

Bavarian prompt:

Schreib an Satz auf Boarisch.

Schreib an Satz auf Boarisch.

Schreib an Satz auf Boarisch.

Schreib an Satz auf Boarisch.

Schreib an Satz auf Boarisch.

Schreib an Satz auf Boarisch.

Schreib an Satz auf Boarisch.

Schreib an Satz auf Boarisch.

Schreib an Satz auf Boarisch.

Schreib an Satz auf Boarisch.

Schreib an Satz auf Boarisch.

Schreib an Satz auf Boarisch.

Schreib an Satz auf Boarisch


In [ ]:
# zero-shot prompting
inputs = tokenizer_betzerl(prompt_qa_zero_de, return_tensors="pt").to(device)
outputs = model_betzerl.generate(**inputs, max_length=128)

generated_text = tokenizer_betzerl.decode(outputs[0], skip_special_tokens=True)
print(generated_text)

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


Generiere eine Frage mit einer indirekten Antwort auf Bairisch.
Annotiere die Polarität der Antwort mit diesem Labelset:
1 = ein klares Ja
2 = ein klares Nein
3 = ja, unter bestimmten Bedingungen
4 = weder ja noch nein (Mitte)
5 = anders, die Antwort passt nicht zur Frage
6 = nicht sicher, ob ja oder nein, weil der Kontext fehlt
7 = nicht sicher, ob ja oder nein, weil der Kontext fehlt
8 = nicht sicher, ob ja oder nein, weil der Kontext fehlt
9 = nicht sicher, ob ja oder nein, weil


In [ ]:
# few-shot prompting
inputs = tokenizer_betzerl(prompt_qa_few_de, return_tensors="pt").to(device)
outputs = model_betzerl.generate(**inputs, max_length=256)

generated_text = tokenizer_betzerl.decode(outputs[0], skip_special_tokens=True)
print(generated_text)

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


Erstelle eine Frage, eine indirekte Antwort und ein korrektes Label nach dem Muster auf Bayerisch.
Verwende nicht die Worte Ja oder Nein oder andere direkte Antworten. 

Verwende diese Labels:
1 = ein klares Ja
2 = ein klares Nein
3 = ja, unter bestimmten Bedingungen
4 = weder ja noch nein (Mitte)
5 = anders, die Antwort passt nicht zur Frage
6 = nicht sicher, ob ja oder nein, weil der Kontext fehlt

Beispiele:
Host du scho vom Finanzamt gheard?	I hob an Briaf vo eana griagt.	1
Host du mei Grawattn gfundn?	I hob ned danach gsuacht.	2
Hoit da Herr Schmid de Vorlesung nächsts Joar?	Der is nächsts Joar nimma an da Uni.	2
Hoit da Herr Schmid de Vorlesung nächsts Joar?	Der is nächsts Joar nimma an da Uni.	2

Verwende diese Labels:
1 = ein klares Ja
2 = ein klares Nein
3 = ja, unter bestimmten Bedingungen
4 = weder ja noch nein (Mitte)


### Gemma

Code from https://huggingface.co/google/gemma-2-2b-it

In [ ]:
model_id_gemma = "google/gemma-2-2b-it"

# load from huggingface
pipe_gemma = pipeline(
    "text-generation",
    model=model_id_gemma,
    model_kwargs={"torch_dtype": torch.bfloat16},
    device="cuda",  # replace with "mps" to run on a Mac device
)

Loading checkpoint shards: 100%|██████████| 2/2 [00:06<00:00,  3.32s/it]
Device set to use cuda


In [6]:
def reformat_prompt_gemma(prompt):
    message = [
        {"role": "user", "content": prompt},
    ]
    return message

In [ ]:
# check general Bavarian capabilities
outputs = pipe_gemma(reformat_prompt_gemma(prompt_test), max_new_tokens=256)
assistant_response = outputs[0]["generated_text"][-1]["content"].strip()
print(assistant_response)

"**Guten Tag, wie geht es dir?**" 

This translates to "Good day, how are you?" in English.


In [ ]:
# zero-shot prompting
outputs = pipe_gemma(reformat_prompt_gemma(prompt_qa_zero), max_new_tokens=256)
assistant_response = outputs[0]["generated_text"][-1]["content"].strip()
print(assistant_response)

**Frage:**  "Hast du schon mal in der Alpenregion Urlaub gemacht?"

**Antwort:** "Ja, aber nur in der Nähe."

**Polarität:** 3 

**Erklärung:** 

The answer is a clear "yes" but with a condition. The person has taken a vacation in the Alps, but not necessarily in the Alps themselves. 


Let me know if you'd like to try another question!


In [ ]:
# few-shot prompting
outputs = pipe_gemma(reformat_prompt_gemma(prompt_qa_few), max_new_tokens=256)
assistant_response = outputs[0]["generated_text"][-1]["content"].strip()
print(assistant_response)

**Question:**  Schallst du da Musik?

**Indirect Answer:**  Da Musik is schon da.

**Label:** 1


### Llama
Code from https://huggingface.co/meta-llama/Llama-3.2-1B-Instruct

In [ ]:
model_id_llama = "meta-llama/Llama-3.2-1B-Instruct"

# load from huggingface
pipe_llama = pipeline(
    "text-generation",
    model=model_id_llama,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)

Device set to use cpu


In [18]:
# variant from EuroLLM's system prompt
system_llama = "You are Llama --- an AI assistant that provides safe, educational and helpful answers."

def reformat_prompt_llama(prompt, system=system_llama):
    message = [
        {"role": "system", "content": system},
        {"role": "user", "content": prompt},
    ]
    return message

In [ ]:
# check general Bavarian capabilities
outputs = pipe_llama(reformat_prompt_llama(prompt_test), max_new_tokens=256)
assistant_response = outputs[0]["generated_text"][-1]["content"].strip()
print(assistant_response)

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Ich hoffe, das hilft: "Ich gehe morgen früh zur Milchweise, um ein paar Eier zu kaufen."


In [ ]:
# zero-shot prompting
outputs = pipe_llama(reformat_prompt_llama(prompt_qa_zero), max_new_tokens=256)
assistant_response = outputs[0]["generated_text"][-1]["content"].strip()
print(assistant_response)

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


**Question:** "Von morgen bis abends sind die Hügel in Bayern so schön, dass man sie nicht mehr verlassen kann."

**Polarity:** 3


In [ ]:
# few-shot prompting
outputs = pipe_llama(reformat_prompt_llama(prompt_qa_few), max_new_tokens=256)
assistant_response = outputs[0]["generated_text"][-1]["content"].strip()
print(assistant_response)

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


1. Hast du ees Gehabt?	I habt ees Gehabt.	3
2. Hast du ees Buch?	I habt ees Buch.	3
3. Hast du ees Sproch?	I habt ees Sproch.	3
4. Hast du ees Geld?	I habt Geld.	3
5. Hast du ees Mudder?	I habt Mudder.	5
6. Hast du ees Zeit?	I habt Zeit.	5


### OLMo
Code from https://huggingface.co/allenai/OLMo-2-1124-7B-Instruct-preview and https://huggingface.co/utter-project/EuroLLM-1.7B-Instruct

In [ ]:
model_id_olmo = "allenai/OLMo-2-1124-7B-Instruct"

# load from huggingface
model_olmo = AutoModelForCausalLM.from_pretrained(model_id_olmo)
tokenizer_olmo = AutoTokenizer.from_pretrained(model_id_olmo)

Loading checkpoint shards: 100%|██████████| 3/3 [04:57<00:00, 99.12s/it] 


In [24]:
def reformat_prompt_olmo(prompt):
    return f"<|endoftext|><|user|>{prompt}<|assistant|>\n>"

In [ ]:
# check general Bavarian capabilities
inputs = tokenizer_olmo(reformat_prompt_olmo(prompt_test), return_tensors="pt")
outputs = model_olmo.generate(**inputs, max_new_tokens=128)

generated_text = tokenizer_olmo.decode(outputs[0], skip_special_tokens=True)
print(generated_text.split("\n")[-1]) # only return the generated response

> Da hör ich schon rein!


In [ ]:
# zero-shot prompting
inputs = tokenizer_olmo(reformat_prompt_olmo(prompt_qa_zero), return_tensors="pt")
outputs = model_olmo.generate(**inputs, max_new_tokens=128)

generated_text = tokenizer_olmo.decode(outputs[0], skip_special_tokens=True)
print(generated_text.split("\n")[-1]) 
# print(generated_text)

The question translates to "Is there a football game at the stadium tonight?" The answer, "D'äs Fussi am Sonntag nicht, aber am Montag däi Spiel drin ist," translates back to "There's no game tonight, but there's one on Monday." This answer is marked as 3 because it indicates there is indeed a game, but it specifies a condition (it's on Monday, not


In [ ]:
# few-shot prompting
inputs = tokenizer_olmo(reformat_prompt_olmo(prompt_qa_few), return_tensors="pt")
outputs = model_olmo.generate(**inputs, max_new_tokens=128)

generated_text = tokenizer_olmo.decode(outputs[0], skip_special_tokens=True)
print(generated_text.split("\n")[-1])

**Label: 4**
